## 1. Imports

In [1]:
import os
import re
import pickle
import warnings

import numpy as np
import pandas as pd

from tqdm.auto import tqdm

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import LatentDirichletAllocation
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.preprocessing import MinMaxScaler

import networkx as nx

warnings.filterwarnings("ignore")

tqdm.pandas()

## 2. Project Paths

In [2]:
PROJECT_PATH = "/content/drive/MyDrive/FinSight_AI"

DATA_PATH = os.path.join(
    PROJECT_PATH,
    "data"
)

PROCESSED_PATH = os.path.join(
    DATA_PATH,
    "processed"
)

TOPIC_PATH = os.path.join(
    PROCESSED_PATH,
    "topics"
)

os.makedirs(
    TOPIC_PATH,
    exist_ok=True
)

## 3. Dataset Paths

In [3]:
FINANCIAL_DATASET = os.path.join(
    PROCESSED_PATH,
    "financial_intelligence",
    "financial_intelligence_dataset.parquet"
)

COMPANY_PROFILES = os.path.join(
    PROCESSED_PATH,
    "company_profiles",
    "company_profiles.parquet"
)

COMPANY_ANALYTICS = os.path.join(
    PROCESSED_PATH,
    "company_analytics",
    "company_analytics.parquet"
)

EVENT_STATISTICS = os.path.join(
    PROCESSED_PATH,
    "event_propagation",
    "event_statistics.parquet"
)

ENTITY_DATASET = os.path.join(
    PROCESSED_PATH,
    "entities",
    "news_entities.parquet"
)

## 4. Load Data

In [4]:
financial_df = pd.read_parquet(
    FINANCIAL_DATASET
)

company_profiles = pd.read_parquet(
    COMPANY_PROFILES
)

analytics_df = pd.read_parquet(
    COMPANY_ANALYTICS
)

event_statistics = pd.read_parquet(
    EVENT_STATISTICS
)

entity_df = pd.read_parquet(
    ENTITY_DATASET)

## 5. Verify

In [5]:
print("Financial Dataset")
print(financial_df.shape)

print()

print("Company Profiles")
print(company_profiles.shape)

print()

print("Company Analytics")
print(analytics_df.shape)

print()

print("Event Statistics")
print(event_statistics.shape)

print()

print("Entities")
print(entity_df.shape)

Financial Dataset
(3215296, 22)

Company Profiles
(6619, 60)

Company Analytics
(6619, 70)

Event Statistics
(30, 19)

Entities
(7345743, 8)


## 6. Keep Required Columns

In [6]:
topic_df = financial_df[
    [
        "news_id",
        "headline",
        "ticker",
        "publisher",
        "published_date",
        "final_event",
        "market_signal",
        "final_confidence"
    ]
].copy()

## 7. Remove Duplicate Headlines

In [7]:
topic_df = (

    topic_df

    .drop_duplicates(

        subset="headline"

    )

    .reset_index(drop=True)

)

## 8. Dataset Summary

In [8]:
summary = pd.DataFrame({

    "Metric":[

        "Unique Headlines",

        "Companies",

        "Publishers",

        "Events"

    ],

    "Value":[

        len(topic_df),

        topic_df["ticker"].nunique(),

        topic_df["publisher"].nunique(),

        topic_df["final_event"].nunique()

    ]

})

display(summary)

,Metric,Value
0,Unique Headlines,1649929
1,Companies,6361
2,Publishers,1047
3,Events,30


## 9. Text Cleaning

In [9]:
def clean_text(text):

    text = str(text).lower()

    text = re.sub(
        r"http\S+",
        " ",
        text
    )

    text = re.sub(
        r"[^a-z0-9 ]",
        " ",
        text
    )

    text = re.sub(
        r"\s+",
        " ",
        text
    )

    return text.strip()

## 10. Apply Cleaning

In [10]:
topic_df["clean_headline"] = (

    topic_df["headline"]

    .progress_apply(

        clean_text

    )

)

  0%|          | 0/1649929 [00:00<?, ?it/s]

## 11. Length Statistics

In [11]:
topic_df["word_count"] = (

    topic_df["clean_headline"]

    .str.split()

    .str.len()

)

display(

    topic_df[

        [

            "headline",

            "clean_headline",

            "word_count"

        ]

    ].head()

)

,headline,clean_headline,word_count
0,Stocks That Hit 52-Week Highs On Friday,stocks that hit 52 week highs on friday,8
1,Stocks That Hit 52-Week Highs On Wednesday,stocks that hit 52 week highs on wednesday,8
2,71 Biggest Movers From Friday,71 biggest movers from friday,5
3,46 Stocks Moving In Friday's Mid-Day Session,46 stocks moving in friday s mid day session,9
4,B of A Securities Maintains Neutral on Agilent...,b of a securities maintains neutral on agilent...,14


## 12. Remove Very Short Headlines

In [12]:
topic_df = topic_df[
    topic_df["word_count"] >= 4
].copy()

print(topic_df.shape)

(1640793, 10)


## 13. Save Intermediate Dataset

In [13]:
topic_df.to_parquet(

    os.path.join(

        TOPIC_PATH,

        "topic_preprocessed.parquet"

    ),

    index=False

)

print("Preprocessed Topic Dataset Saved")

Preprocessed Topic Dataset Saved


## 14. Prepare Text Corpus

In [14]:
corpus = topic_df["clean_headline"].fillna("").tolist()

print("Total Headlines :", len(corpus))

Total Headlines : 1640793


## 15. Create Training Sample

In [15]:
SAMPLE_SIZE = min(200000, len(topic_df))

train_df = topic_df.sample(

    n=SAMPLE_SIZE,

    random_state=42

).reset_index(drop=True)

print(train_df.shape)

(200000, 10)


## 16. TF-IDF Vectorizer

In [16]:
vectorizer = TfidfVectorizer(

    max_features=5000,

    stop_words="english",

    ngram_range=(1,2),

    min_df=10,

    max_df=0.80

)

## 17. Fit TF-IDF

In [17]:
X_train = vectorizer.fit_transform(

    train_df["clean_headline"]

)

print(X_train.shape)

(200000, 5000)


## 18. Train LDA

In [18]:
NUM_TOPICS = 20

lda = LatentDirichletAllocation(

    n_components=NUM_TOPICS,

    random_state=42,

    learning_method="batch",

    max_iter=20,

    n_jobs=-1

)

lda.fit(X_train)

print("LDA Training Complete")

LDA Training Complete


In [19]:
import pickle

with open(

    os.path.join(

        TOPIC_PATH,

        "lda_model.pkl"

    ),

    "wb"

) as f:

    pickle.dump(lda, f)

with open(

    os.path.join(

        TOPIC_PATH,

        "tfidf_vectorizer.pkl"

    ),

    "wb"

) as f:

    pickle.dump(vectorizer, f)

print("LDA model and TF-IDF vectorizer saved.")

LDA model and TF-IDF vectorizer saved.


## 19. Display Top Keywords

In [20]:
feature_names = vectorizer.get_feature_names_out()

topic_keywords = []

for topic_idx, topic in enumerate(lda.components_):

    top_words = [

        feature_names[i]

        for i in topic.argsort()[:-16:-1]

    ]

    topic_keywords.append({

        "topic_id": topic_idx,

        "keywords": ", ".join(top_words)

    })

topic_keywords = pd.DataFrame(topic_keywords)

display(topic_keywords)

,topic_id,keywords
0,0,"agreement, year, dollar, announces, rising, be..."
1,1,"week, high, 52, 52 week, stocks, hits, communi..."
2,2,"analyst, blog, analyst blog, stanley, morgan s..."
3,3,"ceo, transcript, earnings transcript, results ..."
4,4,"beats, initiates, coverage, initiates coverage..."
5,5,"dividend, declares, value, stock, investors, p..."
6,6,"fda, phase, study, pharma, data, pharmaceutica..."
7,7,"buys, management, sells, capital, acquires, ll..."
8,8,"stocks, market, stock, china, cap, industry, h..."
9,9,"oil, contract, gold, gas, energy, new, natural..."


## 20. Assign Topics to Training Sample

In [21]:
topic_prob = lda.transform(X_train)

train_df["topic_id"] = topic_prob.argmax(axis=1)

train_df["topic_confidence"] = topic_prob.max(axis=1)

display(

    train_df[

        [

            "headline",

            "topic_id",

            "topic_confidence"

        ]

    ].head()

)

,headline,topic_id,topic_confidence
0,"BOS Better Online Solns Q2 EPS $0.05, Same YoY...",16,0.683794
1,ServiceNow (NOW) to Report Q4 Earnings: What's...,18,0.730279
2,"AGCO beats by $0.23, beats on revenue",4,0.699082
3,J.P. Morgan Reports Solid 1Q Results For The C...,2,0.731087
4,"Gilead - Medivation Too Costly, How About Smal...",6,0.503292


## 21. Topic Distribution

In [22]:
topic_distribution = (

    train_df

    .groupby("topic_id")

    .size()

    .reset_index(name="articles")

    .sort_values(

        "articles",

        ascending=False

    )

)

display(topic_distribution)

,topic_id,articles
16,16,19388
8,8,15863
13,13,15264
9,9,12842
19,19,12715
18,18,12606
5,5,11261
3,3,9776
2,2,9731
4,4,9627


## 22. Average Topic Confidence

In [23]:
topic_confidence = (

    train_df

    .groupby("topic_id")["topic_confidence"]

    .mean()

    .reset_index()

)

display(topic_confidence)

,topic_id,topic_confidence
0,0,0.514960
1,1,0.538333
2,2,0.525231
3,3,0.621764
4,4,0.610722
5,5,0.514730
6,6,0.534284
7,7,0.543702
8,8,0.559584
9,9,0.530937


## 23. Merge Topic Statistics

In [24]:
topic_profiles = (

    topic_distribution

    .merge(

        topic_confidence,

        on="topic_id"

    )

    .merge(

        topic_keywords,

        on="topic_id"

    )

)

display(topic_profiles)

,topic_id,articles,topic_confidence,keywords
0,16,19388,0.729797,"vs, est, eps, reports, sales, estimate, sees, ..."
1,8,15863,0.559584,"stocks, market, stock, china, cap, industry, h..."
2,13,15264,0.533561,"says, term, long, market, hearing, shares, dea..."
3,9,12842,0.530937,"oil, contract, gold, gas, energy, new, natural..."
4,19,12715,0.577258,"maintains, raises, target, pt, price, price ta..."
5,18,12606,0.582837,"earnings, estimates, shares, beat, offering, q..."
6,5,11261,0.514730,"dividend, declares, value, stock, investors, p..."
7,3,9776,0.621764,"ceo, transcript, earnings transcript, results ..."
8,2,9731,0.525231,"analyst, blog, analyst blog, stanley, morgan s..."
9,4,9627,0.610722,"beats, initiates, coverage, initiates coverage..."


## 24. Financial Topic Naming Dictionary

In [25]:
TOPIC_NAME_RULES = {

    "Earnings & Estimates": [
        "eps",
        "earnings",
        "estimate",
        "est",
        "quarter",
        "revenue",
        "sales"
    ],

    "Analyst Ratings": [
        "analyst",
        "price",
        "target",
        "raises",
        "maintains",
        "downgrade",
        "upgrade",
        "pt"
    ],

    "Market Movement": [
        "stocks",
        "market",
        "shares",
        "52",
        "high",
        "watch"
    ],

    "Energy & Commodities": [
        "oil",
        "gas",
        "gold",
        "energy",
        "commodity",
        "natural"
    ],

    "Healthcare & Pharma": [
        "fda",
        "drug",
        "trial",
        "study",
        "phase",
        "pharma"
    ],

    "Dividends & Shareholder Returns": [
        "dividend",
        "investors",
        "yield",
        "buyback"
    ],

    "Corporate Actions": [
        "acquires",
        "acquisition",
        "agreement",
        "management",
        "capital",
        "merger"
    ],

    "Executive Updates": [
        "ceo",
        "executive",
        "management",
        "conference"
    ],

    "Financial Institutions": [
        "bank",
        "morgan",
        "deutsche",
        "jp"
    ],

    "News & Commentary": [
        "benzinga",
        "alert",
        "blog",
        "investor"
    ]
}

## 25. Automatic Topic Naming Function

In [26]:
def generate_topic_name(keyword_string):

    keyword_string = keyword_string.lower()

    best_name = "General Finance"

    best_score = 0

    for topic_name, words in TOPIC_NAME_RULES.items():

        score = sum(

            word in keyword_string

            for word in words

        )

        if score > best_score:

            best_score = score

            best_name = topic_name

    return best_name

## 26. Assign Topic Names

In [27]:
topic_profiles["topic_name"] = (

    topic_profiles["keywords"]

    .apply(generate_topic_name)

)

## 27. Remove Duplicate Topic Names

In [28]:
topic_profiles["topic_name"] = (

    topic_profiles

    .groupby("topic_name")

    .cumcount()

    .astype(str)

    .radd(

        topic_profiles["topic_name"]

        + " "

    )

)

## 28. Reorder Columns

In [29]:
topic_profiles = topic_profiles[
    [
        "topic_id",
        "topic_name",
        "articles",
        "topic_confidence",
        "keywords"
    ]
]

## 29. View Topic Profiles

In [30]:
display(topic_profiles)

,topic_id,topic_name,articles,topic_confidence,keywords
0,16,Earnings & Estimates 0,19388,0.729797,"vs, est, eps, reports, sales, estimate, sees, ..."
1,8,Market Movement 0,15863,0.559584,"stocks, market, stock, china, cap, industry, h..."
2,13,Market Movement 1,15264,0.533561,"says, term, long, market, hearing, shares, dea..."
3,9,Energy & Commodities 0,12842,0.530937,"oil, contract, gold, gas, energy, new, natural..."
4,19,Analyst Ratings 0,12715,0.577258,"maintains, raises, target, pt, price, price ta..."
5,18,Earnings & Estimates 1,12606,0.582837,"earnings, estimates, shares, beat, offering, q..."
6,5,Dividends & Shareholder Returns 0,11261,0.514730,"dividend, declares, value, stock, investors, p..."
7,3,Earnings & Estimates 2,9776,0.621764,"ceo, transcript, earnings transcript, results ..."
8,2,Earnings & Estimates 3,9731,0.525231,"analyst, blog, analyst blog, stanley, morgan s..."
9,4,Earnings & Estimates 4,9627,0.610722,"beats, initiates, coverage, initiates coverage..."


## 30. Save Topic Profiles

In [31]:
topic_profiles.to_parquet(

    os.path.join(

        TOPIC_PATH,

        "topic_profiles.parquet"

    ),

    index=False

)

## 31. Assign Topics to All Headlines

In [32]:
BATCH_SIZE = 50000

topic_ids = []

topic_confidences = []

for start in tqdm(range(0, len(topic_df), BATCH_SIZE)):

    end = min(start + BATCH_SIZE, len(topic_df))

    batch = topic_df.iloc[start:end]

    X_batch = vectorizer.transform(

        batch["clean_headline"]

    )

    probs = lda.transform(X_batch)

    topic_ids.extend(

        probs.argmax(axis=1)

    )

    topic_confidences.extend(

        probs.max(axis=1)

    )

  0%|          | 0/33 [00:00<?, ?it/s]

## 32. Add Topics Back

In [33]:
topic_df["topic_id"] = topic_ids

topic_df["topic_confidence"] = topic_confidences

print(topic_df.shape)

display(
    topic_df.head()
)

(1640793, 12)


,news_id,headline,ticker,publisher,published_date,final_event,market_signal,final_confidence,clean_headline,word_count,topic_id,topic_confidence
0,1,Stocks That Hit 52-Week Highs On Friday,A,Benzinga Insights,2020-06-05 14:30:54+00:00,Market Movement,Neutral,1.0,stocks that hit 52 week highs on friday,8,1,0.748863
1,2,Stocks That Hit 52-Week Highs On Wednesday,A,Benzinga Insights,2020-06-03 14:45:20+00:00,Market Movement,Neutral,1.0,stocks that hit 52 week highs on wednesday,8,1,0.748856
2,3,71 Biggest Movers From Friday,A,Lisa Levin,2020-05-26 08:30:07+00:00,Market Movement,Neutral,1.0,71 biggest movers from friday,5,10,0.520242
3,4,46 Stocks Moving In Friday's Mid-Day Session,A,Lisa Levin,2020-05-22 16:45:06+00:00,Market Movement,Neutral,1.0,46 stocks moving in friday s mid day session,9,15,0.477931
4,5,B of A Securities Maintains Neutral on Agilent...,A,Vick Meyer,2020-05-22 15:38:59+00:00,Analyst Rating,Bullish,1.0,b of a securities maintains neutral on agilent...,14,19,0.788773


## 33. Merge Topic Names

In [34]:
topic_df = topic_df.merge(

    topic_profiles[
        [
            "topic_id",
            "topic_name"
        ]
    ],

    on="topic_id",

    how="left"

)

## 34. Topic Popularity

In [35]:
topic_popularity = (

    topic_df

    .groupby(

        [

            "topic_id",

            "topic_name"

        ]

    )

    .agg(

        articles=("headline","count"),

        companies=("ticker","nunique"),

        publishers=("publisher","nunique"),

        events=("final_event","nunique"),

        avg_confidence=("topic_confidence","mean")

    )

    .reset_index()

)

## 35. Normalize Scores

In [36]:
from sklearn.preprocessing import MinMaxScaler

scaler = MinMaxScaler()

topic_popularity[

    [

        "article_score",

        "company_score",

        "publisher_score",

        "event_score",

        "confidence_score"

    ]

] = scaler.fit_transform(

    topic_popularity[

        [

            "articles",

            "companies",

            "publishers",

            "events",

            "avg_confidence"

        ]

    ]

)

## 36. Popularity Score

In [37]:
topic_popularity["popularity_score"] = (

      0.35 * topic_popularity["article_score"]

    + 0.25 * topic_popularity["company_score"]

    + 0.15 * topic_popularity["publisher_score"]

    + 0.15 * topic_popularity["event_score"]

    + 0.10 * topic_popularity["confidence_score"]

) * 100

## 37. Sort

In [38]:
topic_popularity = topic_popularity.sort_values(

    "popularity_score",

    ascending=False

).reset_index(drop=True)

## 38. Display

In [39]:
display(

    topic_popularity[
        [

            "topic_name",

            "articles",

            "companies",

            "publishers",

            "events",

            "avg_confidence",

            "popularity_score"

        ]
    ]

)

,topic_name,articles,companies,publishers,events,avg_confidence,popularity_score
0,Market Movement 0,132316,5126,663,30,0.551522,63.246405
1,Earnings & Estimates 0,159573,4565,182,30,0.726388,54.399571
2,Market Movement 1,127601,4877,583,30,0.524531,53.650224
3,Dividends & Shareholder Returns 0,91106,5438,490,30,0.508244,48.929279
4,Energy & Commodities 0,104813,4780,528,30,0.520173,43.066876
5,Earnings & Estimates 1,102383,4637,317,30,0.576565,35.732434
6,Corporate Actions 0,75195,5030,424,30,0.504450,34.564648
7,Earnings & Estimates 2,80318,4584,285,30,0.613960,28.754049
8,Analyst Ratings 0,105277,4131,312,30,0.570797,27.155624
9,Corporate Actions 1,61446,4857,355,30,0.533882,26.461249


## 39. Save

In [40]:
topic_popularity.to_parquet(

    os.path.join(

        TOPIC_PATH,

        "topic_popularity.parquet"

    ),

    index=False

)

## 40. Create Monthly Timeline

In [41]:
topic_df["month"] = (

    pd.to_datetime(

        topic_df["published_date"]

    )

    .dt.to_period("M")

    .astype(str)

)

topic_monthly = (

    topic_df

    .groupby(

        [

            "topic_id",

            "topic_name",

            "month"

        ]

    )

    .size()

    .reset_index(

        name="articles"

    )

)

display(topic_monthly.head())

,topic_id,topic_name,month,articles
0,0,Corporate Actions 0,2009-06,2
1,0,Corporate Actions 0,2009-07,4
2,0,Corporate Actions 0,2009-08,116
3,0,Corporate Actions 0,2009-09,33
4,0,Corporate Actions 0,2009-10,28


## 41. Calculate Previous Month Articles

In [42]:
topic_monthly = (

    topic_monthly

    .sort_values(

        [

            "topic_id",

            "month"

        ]

    )

)

topic_monthly["previous_articles"] = (

    topic_monthly

    .groupby("topic_id")["articles"]

    .shift(1)

)

## 42. Monthly Growth %

In [43]:
topic_monthly["growth_rate"] = (

    (

        topic_monthly["articles"]

        -

        topic_monthly["previous_articles"]

    )

    /

    topic_monthly["previous_articles"]

) * 100

topic_monthly["growth_rate"] = (

    topic_monthly["growth_rate"]

    .replace(

        [

            np.inf,

            -np.inf

        ],

        np.nan

    )

)

## 43. Average Growth Per Topic

In [44]:
topic_growth = (

    topic_monthly

    .groupby(

        [

            "topic_id",

            "topic_name"

        ]

    )

    .agg(

        avg_growth=(

            "growth_rate",

            "mean"

        ),

        max_growth=(

            "growth_rate",

            "max"

        ),

        min_growth=(

            "growth_rate",

            "min"

        )

    )

    .reset_index()

)

## 44. Growth Score

In [45]:
scaler = MinMaxScaler()

topic_growth["growth_score"] = scaler.fit_transform(

    topic_growth[

        [

            "avg_growth"

        ]

    ].fillna(0)

) * 100

## 45. Topic Lifecycle

In [46]:
def topic_stage(score):

    if score >= 75:

        return "Emerging"

    elif score >= 55:

        return "Growing"

    elif score >= 35:

        return "Stable"

    else:

        return "Declining"

## 46. Assign Lifecycle

In [47]:
topic_growth["lifecycle"] = (

    topic_growth["growth_score"]

    .apply(

        topic_stage

    )

)

## 47. Sort

In [48]:
topic_growth = topic_growth.sort_values(

    "growth_score",

    ascending=False

)

display(topic_growth)

,topic_id,topic_name,avg_growth,max_growth,min_growth,growth_score,lifecycle
8,8,Market Movement 0,102.432316,13100.000000,-71.656051,100.000000,Emerging
16,16,Earnings & Estimates 0,59.895858,1400.000000,-90.311005,54.698497,Stable
5,5,Dividends & Shareholder Returns 0,57.887297,6900.000000,-87.388535,52.559370,Stable
3,3,Earnings & Estimates 2,47.765068,4400.000000,-83.662281,41.779155,Stable
17,17,Earnings & Estimates 5,43.751953,2000.000000,-86.946387,37.505171,Stable
18,18,Earnings & Estimates 1,41.850941,3700.000000,-83.234501,35.480585,Stable
6,6,Healthcare & Pharma 0,33.076144,3750.000000,-67.679222,26.135391,Declining
9,9,Energy & Commodities 0,29.693953,2775.000000,-76.180258,22.533343,Declining
13,13,Market Movement 1,26.536069,2750.000000,-78.723404,19.170184,Declining
11,11,Earnings & Estimates 7,26.324821,1716.666667,-90.825688,18.945204,Declining


## 48. Merge Into Topic Profiles

In [49]:
topic_profiles = topic_profiles.merge(

    topic_growth[

        [

            "topic_id",

            "growth_score",

            "lifecycle"

        ]

    ],

    on="topic_id",

    how="left"

)

display(topic_profiles.head())

,topic_id,topic_name,articles,topic_confidence,keywords,growth_score,lifecycle
0,16,Earnings & Estimates 0,19388,0.729797,"vs, est, eps, reports, sales, estimate, sees, ...",54.698497,Stable
1,8,Market Movement 0,15863,0.559584,"stocks, market, stock, china, cap, industry, h...",100.000000,Emerging
2,13,Market Movement 1,15264,0.533561,"says, term, long, market, hearing, shares, dea...",19.170184,Declining
3,9,Energy & Commodities 0,12842,0.530937,"oil, contract, gold, gas, energy, new, natural...",22.533343,Declining
4,19,Analyst Ratings 0,12715,0.577258,"maintains, raises, target, pt, price, price ta...",0.000000,Declining


## 49. Save

In [50]:
topic_growth.to_parquet(

    os.path.join(

        TOPIC_PATH,

        "topic_growth.parquet"

    ),

    index=False

)

# 6. Topic Diversity Engine

## 50. Company Diversity

In [51]:
company_diversity = (

    topic_df

    .groupby(

        [

            "topic_id",

            "topic_name"

        ]

    )["ticker"]

    .nunique()

    .reset_index(

        name="company_diversity"

    )

)

display(company_diversity.head())

,topic_id,topic_name,company_diversity
0,0,Corporate Actions 0,5030
1,1,Market Movement 2,4682
2,2,Earnings & Estimates 3,4290
3,3,Earnings & Estimates 2,4584
4,4,Earnings & Estimates 4,4205


## 51. Publisher Diversity

In [52]:
publisher_diversity = (

    topic_df

    .groupby(

        [

            "topic_id",

            "topic_name"

        ]

    )["publisher"]

    .nunique()

    .reset_index(

        name="publisher_diversity"

    )

)

display(publisher_diversity.head())

,topic_id,topic_name,publisher_diversity
0,0,Corporate Actions 0,424
1,1,Market Movement 2,431
2,2,Earnings & Estimates 3,314
3,3,Earnings & Estimates 2,285
4,4,Earnings & Estimates 4,254


## 52. Event Diversity

In [53]:
event_diversity = (

    topic_df

    .groupby(

        [

            "topic_id",

            "topic_name"

        ]

    )["final_event"]

    .nunique()

    .reset_index(

        name="event_diversity"

    )

)

display(event_diversity.head())

,topic_id,topic_name,event_diversity
0,0,Corporate Actions 0,30
1,1,Market Movement 2,30
2,2,Earnings & Estimates 3,30
3,3,Earnings & Estimates 2,30
4,4,Earnings & Estimates 4,30


## 53. Merge Diversity Tables

In [54]:
topic_diversity = (

    company_diversity

    .merge(

        publisher_diversity,

        on=[

            "topic_id",

            "topic_name"

        ]

    )

    .merge(

        event_diversity,

        on=[

            "topic_id",

            "topic_name"

        ]

    )

)

display(topic_diversity.head())

,topic_id,topic_name,company_diversity,publisher_diversity,event_diversity
0,0,Corporate Actions 0,5030,424,30
1,1,Market Movement 2,4682,431,30
2,2,Earnings & Estimates 3,4290,314,30
3,3,Earnings & Estimates 2,4584,285,30
4,4,Earnings & Estimates 4,4205,254,30


## 54. Normalize Diversity

In [55]:
scaler = MinMaxScaler()

topic_diversity[

    [

        "company_score",

        "publisher_score",

        "event_score"

    ]

] = scaler.fit_transform(

    topic_diversity[

        [

            "company_diversity",

            "publisher_diversity",

            "event_diversity"

        ]

    ]

)

## 55. Diversity Score

In [56]:
topic_diversity["diversity_score"] = (

      0.50 * topic_diversity["company_score"]

    + 0.30 * topic_diversity["publisher_score"]

    + 0.20 * topic_diversity["event_score"]

) * 100

## 56. Diversity Category

In [57]:
def diversity_level(score):

    if score >= 80:

        return "Highly Diverse"

    elif score >= 60:

        return "Broad Coverage"

    elif score >= 40:

        return "Moderately Diverse"

    else:

        return "Focused"

## 57. Assign Diversity Label

In [58]:
topic_diversity["diversity_category"] = (

    topic_diversity["diversity_score"]

    .apply(

        diversity_level

    )

)

## 58. Sort

In [59]:
topic_diversity = topic_diversity.sort_values(

    "diversity_score",

    ascending=False

)

display(topic_diversity)

,topic_id,topic_name,company_diversity,publisher_diversity,event_diversity,company_score,publisher_score,event_score,diversity_score,diversity_category
5,5,Dividends & Shareholder Returns 0,5438,490,30,1.000000,0.640333,0.0,69.209979,Broad Coverage
8,8,Market Movement 0,5126,663,30,0.776984,1.000000,0.0,68.849178,Broad Coverage
13,13,Market Movement 1,4877,583,30,0.598999,0.833680,0.0,54.960359,Moderately Diverse
0,0,Corporate Actions 0,5030,424,30,0.708363,0.503119,0.0,50.511711,Moderately Diverse
9,9,Energy & Commodities 0,4780,528,30,0.529664,0.719335,0.0,48.063244,Moderately Diverse
7,7,Corporate Actions 1,4857,355,30,0.584703,0.359667,0.0,40.025189,Moderately Diverse
1,1,Market Movement 2,4682,431,30,0.459614,0.517672,0.0,38.510846,Focused
14,14,Market Movement 3,4700,375,30,0.472480,0.401247,0.0,35.661439,Focused
18,18,Earnings & Estimates 1,4637,317,30,0.427448,0.280665,0.0,29.792367,Focused
3,3,Earnings & Estimates 2,4584,285,30,0.389564,0.214137,0.0,25.902315,Focused


## 59. Merge With Topic Profiles

In [60]:
topic_profiles = topic_profiles.merge(

    topic_diversity[

        [

            "topic_id",

            "diversity_score",

            "diversity_category"

        ]

    ],

    on="topic_id",

    how="left"

)

display(topic_profiles.head())

,topic_id,topic_name,articles,topic_confidence,keywords,growth_score,lifecycle,diversity_score,diversity_category
0,16,Earnings & Estimates 0,19388,0.729797,"vs, est, eps, reports, sales, estimate, sees, ...",54.698497,Stable,18.799142,Focused
1,8,Market Movement 0,15863,0.559584,"stocks, market, stock, china, cap, industry, h...",100.000000,Emerging,68.849178,Broad Coverage
2,13,Market Movement 1,15264,0.533561,"says, term, long, market, hearing, shares, dea...",19.170184,Declining,54.960359,Moderately Diverse
3,9,Energy & Commodities 0,12842,0.530937,"oil, contract, gold, gas, energy, new, natural...",22.533343,Declining,48.063244,Moderately Diverse
4,19,Analyst Ratings 0,12715,0.577258,"maintains, raises, target, pt, price, price ta...",0.000000,Declining,11.396171,Focused


## 60. Save

In [61]:
topic_diversity.to_parquet(

    os.path.join(

        TOPIC_PATH,

        "topic_diversity.parquet"

    ),

    index=False

)

## 61. Topic × Company Matrix

In [62]:
topic_company_matrix = (

    topic_df

    .groupby(

        [

            "topic_id",

            "ticker"

        ]

    )

    .size()

    .unstack(

        fill_value=0

    )

)

print(topic_company_matrix.shape)

display(topic_company_matrix.iloc[:5, :5])

(20, 6619)


ticker,A,AA,AAC,AADR,AAL
topic_id,,,,,
0,74,51,3,2,28
1,60,129,15,0,17
2,140,137,8,0,10
3,65,19,25,0,8
4,59,57,19,0,30


## 62. Topic Similarity Matrix

In [63]:
topic_similarity_matrix = cosine_similarity(

    topic_company_matrix

)

topic_similarity_matrix = pd.DataFrame(

    topic_similarity_matrix,

    index=topic_company_matrix.index,

    columns=topic_company_matrix.index

)

display(

    topic_similarity_matrix.iloc[:5, :5]

)

topic_id,0,1,2,3,4
topic_id,,,,,
0,1.000000,0.395507,0.353405,0.360527,0.325985
1,0.395507,1.000000,0.542100,0.508270,0.508596
2,0.353405,0.542100,1.000000,0.548219,0.511550
3,0.360527,0.508270,0.548219,1.000000,0.592069
4,0.325985,0.508596,0.511550,0.592069,1.000000


## 63. Extract Similar Topic Pairs

In [64]:
similar_topics = []

topic_ids = topic_similarity_matrix.index.tolist()

for i in range(len(topic_ids)):

    for j in range(i + 1, len(topic_ids)):

        score = topic_similarity_matrix.iloc[i, j]

        if score >= 0.50:

            similar_topics.append({

                "topic_id_1": topic_ids[i],

                "topic_id_2": topic_ids[j],

                "similarity": round(score, 4)

            })

similar_topics = pd.DataFrame(similar_topics)

display(similar_topics.head(20))

,topic_id_1,topic_id_2,similarity
0,1,2,0.5421
1,1,3,0.5083
2,1,4,0.5086
3,1,8,0.5422
4,1,13,0.5592
5,1,14,0.5695
6,1,16,0.5699
7,1,18,0.6569
8,1,19,0.5674
9,2,3,0.5482


## 64. Attach Topic Names

In [65]:
topic_name_lookup = (

    topic_profiles

    .set_index("topic_id")["topic_name"]

    .to_dict()

)

similar_topics["topic_1"] = (

    similar_topics["topic_id_1"]

    .map(topic_name_lookup)

)

similar_topics["topic_2"] = (

    similar_topics["topic_id_2"]

    .map(topic_name_lookup)

)

similar_topics = similar_topics[

    [

        "topic_1",

        "topic_2",

        "similarity"

    ]

]

display(similar_topics.head(20))

,topic_1,topic_2,similarity
0,Market Movement 2,Earnings & Estimates 3,0.5421
1,Market Movement 2,Earnings & Estimates 2,0.5083
2,Market Movement 2,Earnings & Estimates 4,0.5086
3,Market Movement 2,Market Movement 0,0.5422
4,Market Movement 2,Market Movement 1,0.5592
5,Market Movement 2,Market Movement 3,0.5695
6,Market Movement 2,Earnings & Estimates 0,0.5699
7,Market Movement 2,Earnings & Estimates 1,0.6569
8,Market Movement 2,Analyst Ratings 0,0.5674
9,Earnings & Estimates 3,Earnings & Estimates 2,0.5482


## 65. Build Topic Graph

In [66]:
G = nx.Graph()

for row in similar_topics.itertuples():

    G.add_edge(

        row.topic_1,

        row.topic_2,

        weight=row.similarity

    )

print("Nodes :", G.number_of_nodes())

print("Edges :", G.number_of_edges())

Nodes : 13
Edges : 41


## 66. Detect Communities

In [67]:
from networkx.algorithms.community import greedy_modularity_communities

communities = list(

    greedy_modularity_communities(

        G

    )

)

print("Communities :", len(communities))

Communities : 3


## 67. Create Community Table

In [68]:
community_rows = []

for community_id, members in enumerate(communities):

    for topic in members:

        community_rows.append({

            "topic_name": topic,

            "community": community_id

        })

topic_communities = pd.DataFrame(

    community_rows

)

display(topic_communities.head())

,topic_name,community
0,Earnings & Estimates 2,0
1,Market Movement 3,0
2,Earnings & Estimates 3,0
3,Analyst Ratings 0,0
4,Earnings & Estimates 0,0


## 68. Merge Community Into Profiles

In [69]:
topic_profiles = topic_profiles.merge(

    topic_communities,

    on="topic_name",

    how="left"

)

topic_profiles["community"] = (

    topic_profiles["community"]

    .fillna(-1)

    .astype(int)

)

display(topic_profiles.head())

,topic_id,topic_name,articles,topic_confidence,keywords,growth_score,lifecycle,diversity_score,diversity_category,community
0,16,Earnings & Estimates 0,19388,0.729797,"vs, est, eps, reports, sales, estimate, sees, ...",54.698497,Stable,18.799142,Focused,0
1,8,Market Movement 0,15863,0.559584,"stocks, market, stock, china, cap, industry, h...",100.000000,Emerging,68.849178,Broad Coverage,1
2,13,Market Movement 1,15264,0.533561,"says, term, long, market, hearing, shares, dea...",19.170184,Declining,54.960359,Moderately Diverse,1
3,9,Energy & Commodities 0,12842,0.530937,"oil, contract, gold, gas, energy, new, natural...",22.533343,Declining,48.063244,Moderately Diverse,-1
4,19,Analyst Ratings 0,12715,0.577258,"maintains, raises, target, pt, price, price ta...",0.000000,Declining,11.396171,Focused,0


## 69. Community Summary

In [70]:
community_summary = (

    topic_profiles

    .groupby("community")

    .agg(

        topics=(

            "topic_name",

            "count"

        ),

        total_articles=(

            "articles",

            "sum"

        ),

        avg_confidence=(

            "topic_confidence",

            "mean"

        )

    )

    .reset_index()

)

display(community_summary)

,community,topics,total_articles,avg_confidence
0,-1,7,54919,0.532627
1,0,7,74845,0.605630
2,1,3,38851,0.543826
3,2,3,31385,0.547090


## 70. Save

In [71]:
similar_topics.to_parquet(

    os.path.join(

        TOPIC_PATH,

        "topic_similarity.parquet"

    ),

    index=False

)

topic_communities.to_parquet(

    os.path.join(

        TOPIC_PATH,

        "topic_communities.parquet"

    ),

    index=False

)

## 71. Impact Category

In [72]:
topic_profiles = topic_profiles.merge(

    topic_popularity[
        [
            "topic_id",
            "popularity_score"
        ]
    ],

    on="topic_id",

    how="left"

)

## 72. Popularity Level

In [73]:
def popularity_level(score):

    if score >= 60:
        return "Very High"

    elif score >= 45:
        return "High"

    elif score >= 25:
        return "Medium"

    else:
        return "Low"

## 73. Assign Popularity Category

In [74]:
topic_profiles["impact_category"] = (

    topic_profiles["popularity_score"]

    .apply(popularity_level)

)

## 74. Confidence Category

In [75]:
def confidence_level(score):

    if score >= 0.70:
        return "Very Reliable"

    elif score >= 0.60:
        return "Reliable"

    elif score >= 0.50:
        return "Moderate"

    else:
        return "Low"

## 75. Assign Confidence Category

In [76]:
topic_profiles["confidence_category"] = (

    topic_profiles["topic_confidence"]

    .apply(confidence_level)

)

## 76. Build Topic Fingerprint

In [77]:
def build_fingerprint(row):

    tags = []

    if row["impact_category"] == "Very High":
        tags.append("High Impact")

    elif row["impact_category"] == "High":
        tags.append("Popular")

    if row["lifecycle"] == "Emerging":
        tags.append("Emerging")

    elif row["lifecycle"] == "Growing":
        tags.append("Growing")

    elif row["lifecycle"] == "Stable":
        tags.append("Stable")

    if row["diversity_category"] == "Highly Diverse":
        tags.append("Highly Diverse")

    elif row["diversity_category"] == "Broad Coverage":
        tags.append("Broad Coverage")

    if row["confidence_category"] == "Very Reliable":
        tags.append("Very Reliable")

    elif row["confidence_category"] == "Reliable":
        tags.append("Reliable")

    return ", ".join(tags)

## 77. Generate Fingerprints

In [78]:
topic_profiles["topic_fingerprint"] = (

    topic_profiles

    .apply(

        build_fingerprint,

        axis=1

    )

)

## 78. Display

In [79]:
display(

    topic_profiles[
        [
            "topic_name",
            "topic_fingerprint"
        ]
    ]

)

,topic_name,topic_fingerprint
0,Earnings & Estimates 0,"Popular, Stable, Very Reliable"
1,Market Movement 0,"High Impact, Emerging, Broad Coverage"
2,Market Movement 1,Popular
3,Energy & Commodities 0,
4,Analyst Ratings 0,
5,Earnings & Estimates 1,Stable
6,Dividends & Shareholder Returns 0,"Popular, Stable, Broad Coverage"
7,Earnings & Estimates 2,"Stable, Reliable"
8,Earnings & Estimates 3,
9,Earnings & Estimates 4,Reliable


## 79. Fingerprint Statistics

In [80]:
fingerprint_summary = (

    topic_profiles

    .groupby("topic_fingerprint")

    .size()

    .reset_index(name="topics")

    .sort_values(

        "topics",

        ascending=False

    )

)

display(fingerprint_summary)

,topic_fingerprint,topics
0,,12
7,"Stable, Reliable",2
2,Popular,1
1,"High Impact, Emerging, Broad Coverage",1
3,"Popular, Stable, Broad Coverage",1
4,"Popular, Stable, Very Reliable",1
5,Reliable,1
6,Stable,1


## 80. Save

In [81]:
fingerprint_summary.to_parquet(

    os.path.join(

        TOPIC_PATH,

        "topic_fingerprints.parquet"

    ),

    index=False

)

## 81. Generate Topic Explanation

In [82]:
def build_topic_summary(row):

    summary = (

        f"Articles: {int(row['articles'])} | "

        f"Confidence: {row['topic_confidence']:.2f} | "

        f"Lifecycle: {row['lifecycle']} | "

        f"Impact: {row['impact_category']} | "

        f"Diversity: {row['diversity_category']}"

    )

    return summary

## 82. Create Summary Column

In [83]:
topic_profiles["summary"] = (

    topic_profiles

    .apply(

        build_topic_summary,

        axis=1

    )

)

## 83. Topic Search Keywords

In [84]:
topic_profiles["search_keywords"] = (

    topic_profiles["keywords"]

    .str.lower()

    .str.replace(",", " ", regex=False)

)

## 84. Display

In [85]:
display(

    topic_profiles[

        [

            "topic_name",

            "summary",

            "search_keywords"

        ]

    ].head()

)

,topic_name,summary,search_keywords
0,Earnings & Estimates 0,Articles: 19388 | Confidence: 0.73 | Lifecycle...,vs est eps reports sales estimate sees ...
1,Market Movement 0,Articles: 15863 | Confidence: 0.56 | Lifecycle...,stocks market stock china cap industry h...
2,Market Movement 1,Articles: 15264 | Confidence: 0.53 | Lifecycle...,says term long market hearing shares dea...
3,Energy & Commodities 0,Articles: 12842 | Confidence: 0.53 | Lifecycle...,oil contract gold gas energy new natural...
4,Analyst Ratings 0,Articles: 12715 | Confidence: 0.58 | Lifecycle...,maintains raises target pt price price ta...


## 85. Topic Search Dictionary

In [86]:
topic_lookup = {}

for row in topic_profiles.itertuples():

    topic_lookup[row.topic_name] = {

        "topic_id": row.topic_id,

        "summary": row.summary,

        "keywords": row.keywords,

        "community": row.community,

        "fingerprint": row.topic_fingerprint

    }

print(len(topic_lookup))

20


## 86. Quick Search Function

In [87]:
def search_topic(name):

    if name in topic_lookup:

        return topic_lookup[name]

    return None

## 87. Test

In [88]:
search_topic(

    topic_profiles.iloc[0]["topic_name"]

)

{'topic_id': 16,
 'summary': 'Articles: 19388 | Confidence: 0.73 | Lifecycle: Stable | Impact: High | Diversity: Focused',
 'keywords': 'vs, est, eps, reports, sales, estimate, sees, q4, q3, adj, q1, q2, vs est, q1 eps, q2 eps',
 'community': 0,
 'fingerprint': 'Popular, Stable, Very Reliable'}

## 88. Recommendation Score

In [89]:
topic_profiles["recommendation_score"] = (

      topic_profiles["popularity_score"] * 0.40

    + topic_profiles["growth_score"] * 0.30

    + topic_profiles["topic_confidence"] * 100 * 0.30

)

## 89. Sort

In [90]:
topic_profiles = (

    topic_profiles

    .sort_values(

        "recommendation_score",

        ascending=False

    )

)

## 90. Display

In [91]:
display(

    topic_profiles[

        [

            "topic_name",

            "recommendation_score"

        ]

    ].head(20)

)

,topic_name,recommendation_score
1,Market Movement 0,72.086074
0,Earnings & Estimates 0,60.063296
6,Dividends & Shareholder Returns 0,50.781419
2,Market Movement 1,43.217971
7,Earnings & Estimates 2,42.688292
5,Earnings & Estimates 1,42.422261
3,Energy & Commodities 0,39.914868
16,Earnings & Estimates 5,37.348573
10,Corporate Actions 0,34.739244
13,Corporate Actions 1,30.985790


## 91. Create Topic Intelligence Cards

In [92]:
topic_cards = topic_profiles[

    [

        "topic_id",

        "topic_name",

        "summary",

        "topic_fingerprint",

        "keywords",

        "articles",

        "topic_confidence",

        "growth_score",

        "recommendation_score",

        "impact_category",

        "lifecycle",

        "diversity_category",

        "community"

    ]

].copy()

display(topic_cards.head())

,topic_id,topic_name,summary,topic_fingerprint,keywords,articles,topic_confidence,growth_score,recommendation_score,impact_category,lifecycle,diversity_category,community
1,8,Market Movement 0,Articles: 15863 | Confidence: 0.56 | Lifecycle...,"High Impact, Emerging, Broad Coverage","stocks, market, stock, china, cap, industry, h...",15863,0.559584,100.000000,72.086074,Very High,Emerging,Broad Coverage,1
0,16,Earnings & Estimates 0,Articles: 19388 | Confidence: 0.73 | Lifecycle...,"Popular, Stable, Very Reliable","vs, est, eps, reports, sales, estimate, sees, ...",19388,0.729797,54.698497,60.063296,High,Stable,Focused,0
6,5,Dividends & Shareholder Returns 0,Articles: 11261 | Confidence: 0.51 | Lifecycle...,"Popular, Stable, Broad Coverage","dividend, declares, value, stock, investors, p...",11261,0.514730,52.559370,50.781419,High,Stable,Broad Coverage,2
2,13,Market Movement 1,Articles: 15264 | Confidence: 0.53 | Lifecycle...,Popular,"says, term, long, market, hearing, shares, dea...",15264,0.533561,19.170184,43.217971,High,Declining,Moderately Diverse,1
7,3,Earnings & Estimates 2,Articles: 9776 | Confidence: 0.62 | Lifecycle:...,"Stable, Reliable","ceo, transcript, earnings transcript, results ...",9776,0.621764,41.779155,42.688292,Medium,Stable,Focused,0


## 92. Rank Topics

In [93]:
topic_cards["overall_rank"] = (

    topic_cards["recommendation_score"]

    .rank(

        ascending=False,

        method="dense"

    )

    .astype(int)

)

## 93. Sort

In [94]:
topic_cards = (

    topic_cards

    .sort_values(

        "overall_rank"

    )

)

## 94. Display

In [95]:
display(

    topic_cards[

        [

            "overall_rank",

            "topic_name",

            "recommendation_score"

        ]

    ]

)

,overall_rank,topic_name,recommendation_score
1,1,Market Movement 0,72.086074
0,2,Earnings & Estimates 0,60.063296
6,3,Dividends & Shareholder Returns 0,50.781419
2,4,Market Movement 1,43.217971
7,5,Earnings & Estimates 2,42.688292
5,6,Earnings & Estimates 1,42.422261
3,7,Energy & Commodities 0,39.914868
16,8,Earnings & Estimates 5,37.348573
10,9,Corporate Actions 0,34.739244
13,10,Corporate Actions 1,30.985790


## 95. Top Recommended Topics

In [96]:
top_topics = (

    topic_cards

    .head(10)

    .copy()

)

display(top_topics)

,topic_id,topic_name,summary,topic_fingerprint,keywords,articles,topic_confidence,growth_score,recommendation_score,impact_category,lifecycle,diversity_category,community,overall_rank
1,8,Market Movement 0,Articles: 15863 | Confidence: 0.56 | Lifecycle...,"High Impact, Emerging, Broad Coverage","stocks, market, stock, china, cap, industry, h...",15863,0.559584,100.000000,72.086074,Very High,Emerging,Broad Coverage,1,1
0,16,Earnings & Estimates 0,Articles: 19388 | Confidence: 0.73 | Lifecycle...,"Popular, Stable, Very Reliable","vs, est, eps, reports, sales, estimate, sees, ...",19388,0.729797,54.698497,60.063296,High,Stable,Focused,0,2
6,5,Dividends & Shareholder Returns 0,Articles: 11261 | Confidence: 0.51 | Lifecycle...,"Popular, Stable, Broad Coverage","dividend, declares, value, stock, investors, p...",11261,0.514730,52.559370,50.781419,High,Stable,Broad Coverage,2,3
2,13,Market Movement 1,Articles: 15264 | Confidence: 0.53 | Lifecycle...,Popular,"says, term, long, market, hearing, shares, dea...",15264,0.533561,19.170184,43.217971,High,Declining,Moderately Diverse,1,4
7,3,Earnings & Estimates 2,Articles: 9776 | Confidence: 0.62 | Lifecycle:...,"Stable, Reliable","ceo, transcript, earnings transcript, results ...",9776,0.621764,41.779155,42.688292,Medium,Stable,Focused,0,5
5,18,Earnings & Estimates 1,Articles: 12606 | Confidence: 0.58 | Lifecycle...,Stable,"earnings, estimates, shares, beat, offering, q...",12606,0.582837,35.480585,42.422261,Medium,Stable,Focused,2,6
3,9,Energy & Commodities 0,Articles: 12842 | Confidence: 0.53 | Lifecycle...,,"oil, contract, gold, gas, energy, new, natural...",12842,0.530937,22.533343,39.914868,Medium,Declining,Moderately Diverse,-1,7
16,17,Earnings & Estimates 5,Articles: 6772 | Confidence: 0.61 | Lifecycle:...,"Stable, Reliable","results, results earnings, earnings, transcrip...",6772,0.613883,37.505171,37.348573,Low,Stable,Focused,0,8
10,0,Corporate Actions 0,Articles: 9120 | Confidence: 0.51 | Lifecycle:...,,"agreement, year, dollar, announces, rising, be...",9120,0.514960,18.215242,34.739244,Medium,Declining,Moderately Diverse,-1,9
13,7,Corporate Actions 1,Articles: 7518 | Confidence: 0.54 | Lifecycle:...,,"buys, management, sells, capital, acquires, ll...",7518,0.543702,13.634130,30.985790,Medium,Declining,Moderately Diverse,2,10


## 96. Recommendation Categories

In [97]:
def recommendation_level(score):

    if score >= 60:

        return "Must Watch"

    elif score >= 45:

        return "Highly Recommended"

    elif score >= 30:

        return "Recommended"

    else:

        return "Monitor"

## 97. Apply Recommendation Labels

In [98]:
topic_cards["recommendation"] = (

    topic_cards["recommendation_score"]

    .apply(

        recommendation_level

    )

)

## 98. Display

In [99]:
display(

    topic_cards[

        [

            "topic_name",

            "recommendation",

            "recommendation_score"

        ]

    ]

)

,topic_name,recommendation,recommendation_score
1,Market Movement 0,Must Watch,72.086074
0,Earnings & Estimates 0,Must Watch,60.063296
6,Dividends & Shareholder Returns 0,Highly Recommended,50.781419
2,Market Movement 1,Recommended,43.217971
7,Earnings & Estimates 2,Recommended,42.688292
5,Earnings & Estimates 1,Recommended,42.422261
3,Energy & Commodities 0,Recommended,39.914868
16,Earnings & Estimates 5,Recommended,37.348573
10,Corporate Actions 0,Recommended,34.739244
13,Corporate Actions 1,Recommended,30.985790


## 99. Recommendation Distribution

In [100]:
recommendation_summary = (

    topic_cards

    .groupby("recommendation")

    .size()

    .reset_index(name="topics")

    .sort_values(

        "topics",

        ascending=False

    )

)

display(recommendation_summary)

,recommendation,topics
1,Monitor,9
3,Recommended,8
2,Must Watch,2
0,Highly Recommended,1


## 100. Topic Recommendation Index

In [101]:
topic_recommendation_index = {

    row.topic_name: {

        "rank": row.overall_rank,

        "recommendation": row.recommendation,

        "score": round(

            row.recommendation_score,

            2

        )

    }

    for row in topic_cards.itertuples()

}

print(len(topic_recommendation_index))

20


## 101. Create Topic Search Index

In [102]:
topic_search_index = {}

for row in topic_cards.itertuples():

    topic_search_index[row.topic_name.lower()] = row.topic_id

    topic_search_index[str(row.topic_id)] = row.topic_id

print(len(topic_search_index))

40


## 102. Create Topic ID Lookup

In [103]:
topic_id_lookup = {}

for row in topic_cards.itertuples():

    topic_id_lookup[row.topic_id] = {

        "name": row.topic_name,

        "rank": row.overall_rank,

        "recommendation": row.recommendation,

        "score": round(row.recommendation_score,2),

        "community": row.community

    }

print(len(topic_id_lookup))

20


## 103. Validation

In [104]:
display(

    topic_cards.head()

)

print()

print(topic_search_index["market movement 0"])

print()

print(topic_id_lookup[16])

,topic_id,topic_name,summary,topic_fingerprint,keywords,articles,topic_confidence,growth_score,recommendation_score,impact_category,lifecycle,diversity_category,community,overall_rank,recommendation
1,8,Market Movement 0,Articles: 15863 | Confidence: 0.56 | Lifecycle...,"High Impact, Emerging, Broad Coverage","stocks, market, stock, china, cap, industry, h...",15863,0.559584,100.000000,72.086074,Very High,Emerging,Broad Coverage,1,1,Must Watch
0,16,Earnings & Estimates 0,Articles: 19388 | Confidence: 0.73 | Lifecycle...,"Popular, Stable, Very Reliable","vs, est, eps, reports, sales, estimate, sees, ...",19388,0.729797,54.698497,60.063296,High,Stable,Focused,0,2,Must Watch
6,5,Dividends & Shareholder Returns 0,Articles: 11261 | Confidence: 0.51 | Lifecycle...,"Popular, Stable, Broad Coverage","dividend, declares, value, stock, investors, p...",11261,0.514730,52.559370,50.781419,High,Stable,Broad Coverage,2,3,Highly Recommended
2,13,Market Movement 1,Articles: 15264 | Confidence: 0.53 | Lifecycle...,Popular,"says, term, long, market, hearing, shares, dea...",15264,0.533561,19.170184,43.217971,High,Declining,Moderately Diverse,1,4,Recommended
7,3,Earnings & Estimates 2,Articles: 9776 | Confidence: 0.62 | Lifecycle:...,"Stable, Reliable","ceo, transcript, earnings transcript, results ...",9776,0.621764,41.779155,42.688292,Medium,Stable,Focused,0,5,Recommended



8

{'name': 'Earnings & Estimates 0', 'rank': 2, 'recommendation': 'Must Watch', 'score': 60.06, 'community': 0}


In [105]:
print(financial_df.columns.tolist())

['news_id', 'headline', 'url', 'publisher', 'published_date', 'ticker', 'source', 'year', 'month', 'day', 'weekday', 'hour', 'headline_length', 'word_count', 'has_time', 'original_headline', 'clean_character_count', 'clean_word_count', 'final_event', 'market_signal', 'final_confidence', 'classification_source']


In [ ]:
print(financial_df.columns.tolist())

In [107]:
news_topics = (

    topic_df[
        [
            "news_id",
            "headline",
            "topic_id",
            "topic_name",
            "topic_confidence"
        ]
    ]

    .copy()

)

print(news_topics.shape)

display(news_topics.head())

(1640793, 5)


,news_id,headline,topic_id,topic_name,topic_confidence
0,1,Stocks That Hit 52-Week Highs On Friday,1,Market Movement 2,0.748863
1,2,Stocks That Hit 52-Week Highs On Wednesday,1,Market Movement 2,0.748856
2,3,71 Biggest Movers From Friday,10,News & Commentary 0,0.520242
3,4,46 Stocks Moving In Friday's Mid-Day Session,15,Earnings & Estimates 6,0.477931
4,5,B of A Securities Maintains Neutral on Agilent...,19,Analyst Ratings 0,0.788773


In [108]:
news_topics.to_parquet(

    os.path.join(

        TOPIC_PATH,

        "news_topics.parquet"

    ),

    index=False

)

print("Saved Successfully")

Saved Successfully


In [109]:
check = pd.read_parquet(

    os.path.join(

        TOPIC_PATH,

        "news_topics.parquet"

    )

)

print(check.shape)

display(check.head())

(1640793, 5)


,news_id,headline,topic_id,topic_name,topic_confidence
0,1,Stocks That Hit 52-Week Highs On Friday,1,Market Movement 2,0.748863
1,2,Stocks That Hit 52-Week Highs On Wednesday,1,Market Movement 2,0.748856
2,3,71 Biggest Movers From Friday,10,News & Commentary 0,0.520242
3,4,46 Stocks Moving In Friday's Mid-Day Session,15,Earnings & Estimates 6,0.477931
4,5,B of A Securities Maintains Neutral on Agilent...,19,Analyst Ratings 0,0.788773


In [120]:
financial_df = financial_df.merge(
    news_topics[
        [
            "headline",
            "topic_id",
            "topic_name",
            "topic_confidence"
        ]
    ],
    on="headline",
    how="left"
)

In [121]:
print(financial_df.shape)

display(
    financial_df[
        [
            "headline",
            "topic_id",
            "topic_name",
            "topic_confidence"
        ]
    ].head()
)

print()

print("Missing Topic IDs :",
      financial_df["topic_id"].isna().sum())

(3215296, 25)


,headline,topic_id,topic_name,topic_confidence
0,Stocks That Hit 52-Week Highs On Friday,1.0,Market Movement 2,0.748863
1,Stocks That Hit 52-Week Highs On Wednesday,1.0,Market Movement 2,0.748856
2,71 Biggest Movers From Friday,10.0,News & Commentary 0,0.520242
3,46 Stocks Moving In Friday's Mid-Day Session,15.0,Earnings & Estimates 6,0.477931
4,B of A Securities Maintains Neutral on Agilent...,19.0,Analyst Ratings 0,0.788773



Missing Topic IDs : 40576


In [122]:
financial_df["topic_id"] = financial_df["topic_id"].fillna(-1).astype(int)

financial_df["topic_name"] = financial_df["topic_name"].fillna("Unknown")

financial_df["topic_confidence"] = financial_df["topic_confidence"].fillna(0)

In [123]:
financial_df.to_parquet(
    FINANCIAL_DATASET,
    index=False
)

print("Financial Intelligence Dataset Updated Successfully")

Financial Intelligence Dataset Updated Successfully


In [124]:
check = pd.read_parquet(FINANCIAL_DATASET)

print(check.shape)

display(
    check[
        [
            "headline",
            "topic_id",
            "topic_name",
            "topic_confidence"
        ]
    ].head()
)

(3215296, 25)


,headline,topic_id,topic_name,topic_confidence
0,Stocks That Hit 52-Week Highs On Friday,1,Market Movement 2,0.748863
1,Stocks That Hit 52-Week Highs On Wednesday,1,Market Movement 2,0.748856
2,71 Biggest Movers From Friday,10,News & Commentary 0,0.520242
3,46 Stocks Moving In Friday's Mid-Day Session,15,Earnings & Estimates 6,0.477931
4,B of A Securities Maintains Neutral on Agilent...,19,Analyst Ratings 0,0.788773


In [125]:
print("Missing Topic IDs :",
      financial_df["topic_id"].isna().sum())

Missing Topic IDs : 0


In [110]:
check = pd.read_parquet(

    os.path.join(

        TOPIC_PATH,

        "news_topics.parquet"

    )

)

print(check.shape)

display(check.head())

(1640793, 5)


,news_id,headline,topic_id,topic_name,topic_confidence
0,1,Stocks That Hit 52-Week Highs On Friday,1,Market Movement 2,0.748863
1,2,Stocks That Hit 52-Week Highs On Wednesday,1,Market Movement 2,0.748856
2,3,71 Biggest Movers From Friday,10,News & Commentary 0,0.520242
3,4,46 Stocks Moving In Friday's Mid-Day Session,15,Earnings & Estimates 6,0.477931
4,5,B of A Securities Maintains Neutral on Agilent...,19,Analyst Ratings 0,0.788773


## 104. Save Main Topic Dataset

In [111]:
topic_cards.to_parquet(

    os.path.join(

        TOPIC_PATH,

        "topic_cards.parquet"

    ),

    index=False

)

## 105. Save Topic Profiles

In [112]:
topic_profiles.to_parquet(

    os.path.join(

        TOPIC_PATH,

        "topic_profiles.parquet"

    ),

    index=False

)

## 106. Save Topic Timeline

In [113]:
topic_monthly.to_parquet(

    os.path.join(

        TOPIC_PATH,

        "topic_timeline.parquet"

    ),

    index=False

)

## 107. Save Topic Popularity

In [114]:
topic_popularity.to_parquet(

    os.path.join(

        TOPIC_PATH,

        "topic_popularity.parquet"

    ),

    index=False

)

## 108. Save Topic Similarity

In [115]:
similar_topics.to_parquet(

    os.path.join(

        TOPIC_PATH,

        "topic_similarity.parquet"

    ),

    index=False

)

## 109. Save Topic Communities

In [116]:
topic_communities.to_parquet(

    os.path.join(

        TOPIC_PATH,

        "topic_communities.parquet"

    ),

    index=False

)

## 110. Save Lookup Dictionaries

In [117]:
import pickle

with open(

    os.path.join(

        TOPIC_PATH,

        "topic_lookup.pkl"

    ),

    "wb"

) as f:

    pickle.dump(

        topic_lookup,

        f

    )

with open(

    os.path.join(

        TOPIC_PATH,

        "topic_search_index.pkl"

    ),

    "wb"

) as f:

    pickle.dump(

        topic_search_index,

        f

    )

with open(

    os.path.join(

        TOPIC_PATH,

        "topic_id_lookup.pkl"

    ),

    "wb"

) as f:

    pickle.dump(

        topic_id_lookup,

        f

    )

### 111. Statistics

In [118]:
summary = pd.DataFrame(

    {

        "Metric":[

            "Topics",

            "Communities",

            "Topic Cards",

            "Topic Timeline",

            "Similarity Links",

            "Search Index",

            "Lookup Objects"

        ],

        "Value":[

            len(topic_cards),

            topic_cards["community"].nunique(),

            len(topic_cards),

            len(topic_monthly),

            len(similar_topics),

            len(topic_search_index),

            len(topic_lookup)

        ]

    }

)

display(summary)

,Metric,Value
0,Topics,20
1,Communities,4
2,Topic Cards,20
3,Topic Timeline,2652
4,Similarity Links,41
5,Search Index,40
6,Lookup Objects,20


### 112. Validation

In [119]:
print("Top Topic")

display(

    topic_cards.head(1)

)

print()

print("Recommendation Distribution")

display(

    recommendation_summary

)

Top Topic


,topic_id,topic_name,summary,topic_fingerprint,keywords,articles,topic_confidence,growth_score,recommendation_score,impact_category,lifecycle,diversity_category,community,overall_rank,recommendation
1,8,Market Movement 0,Articles: 15863 | Confidence: 0.56 | Lifecycle...,"High Impact, Emerging, Broad Coverage","stocks, market, stock, china, cap, industry, h...",15863,0.559584,100.0,72.086074,Very High,Emerging,Broad Coverage,1,1,Must Watch



Recommendation Distribution


,recommendation,topics
1,Monitor,9
3,Recommended,8
2,Must Watch,2
0,Highly Recommended,1
